# 02 · EDA: risk, price levels, cost per unit, supply chain, selling prices

What this notebook answers, for the 3 / 6 / 9-month forecasts and the quote buffer:

0. **Two cost indices**: the PPI-proxy index and the rebuilt market-price index.
1. **Stationarity and tails**: what to model, and why the backtest's 80% bands come out too narrow.
2. **Cost-index rises**: how much the material cost of one unit rose in 3 / 6 / 9 months, and which material drove it.
3. **Regimes**: six periods since 2016 and how volatile each one was.
4. **Co-movement**: how material prices move together, whether that changes in a crisis, and why summing P90s is wrong.
5. **Key material prices**: where copper, steel, stainless and plastics stand against 2017 and their 2022 peaks.
6. **Material cost per unit**: the cost index in EUR per unit over time, its 12-month change and the RM6 - AirSeT gap.
7. **Price risk by horizon**: the typical move of every series over 3 / 6 / 9 months.
8. **Supply-chain transmission**: which upstream prices move ahead of the materials Schneider buys.
9. **Selling-price benchmark**: RMU selling prices by brand, and how material cost compares.

All computations are in `src/eda_risk.py` (§1–4), `src/eda.py` (§5–7, §9) and `src/diagrams.py` (§8); the pipeline
saves the same tables and figures to `outputs/eda/`. This notebook shows them and draws the conclusions. Window: Nov-2016 to Sep-2026. A "change" is a log change of a monthly
average price.

> Outputs show challenge data. The pre-commit hook (nbstripout) removes them from every commit.

In [ ]:
import logging
import sys
sys.path.insert(0, "../src")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

import numpy as np
import pandas as pd

import data as D
import diagrams as G
import eda as E
import eda_risk as R

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

df, prices, costs = R.load()  # studied series; full preprocessed panel; BOM cost table
panel = D.panel(bom_file=D.BOM_FILE_V1)  # BOM targets (incl. USD prices) + both versions of the two cost indices
pct = lambda t: (100 * t).round(1)

## 0. Two versions of the material cost index

Both use the same fixed weights: kg per unit (BOM workbook `RM_Prices_XHEC_Challenge_v1.xlsx`, which
replaces the older BOM in `data.zip`) × EUR/t in Sep-2026. They differ only in the price series that
moves the steel lines:

- **PPI proxy** (`RM6_NE_IQI_433520`, `RMAirSeT_IIQ_24kV`): carbon and stainless steel follow the EU steel PPI (S09),
  galvanised steel the German steel PPI (S10), the fallback in `bom_materials.csv`.
- **Market prices** (`…_mkt`): the same lines follow the CRU EUR/t prices that set their weight (CR coil, stainless
  304, HDG coil), the BOM file's first choice (L01–L03). Stainless ends Aug-2026, so its last value is carried to
  Sep-2026 (`data.TRAIL_FILL`).

Copper, aluminium, plastics and epoxy (which moves with the plastics PPI) are the same in both.

In [ ]:
t = costs.assign(market_series=[D.MARKET_PROXY.get(m, p) for m, p in zip(costs.material, costs.proxy)])
t[["product", "material", "kg", "eur_per_unit", "cost_share", "proxy", "market_series"]].round(3)

The PPI proxy drives 50% of the AirSeT index (carbon and stainless steel) and 30% of the RM6 index. The
data-quality notebook showed that S09 lags CR coil by about a month and swings only 0.63× as much, so this choice
matters most for AirSeT.

## 1. Stationarity, tails and volatility clustering

### 1.1 Unit roots
ADF tests H0 = unit root, KPSS tests H0 = stationary. "Unit root" = ADF does not reject and KPSS rejects.

In [ ]:
R.stationarity(df).round(3)

- **Price levels have a unit root**: 17 of 22 series clearly; the other 5 (zinc, the three CRU steel prices, EUR/USD)
  are unclear because neither test rejects with ~120 points. No level is clearly stationary.
- **Monthly changes are stationary** for 18 of 22. The 4 borderline ones (copper and aluminium PPIs, both RM6 indices)
  have ADF p = 0.06–0.09 and KPSS does not reject.
- So models work on **log changes**, not levels, as the backtest already does. The P10–P90 range is a range of
  changes from today's price.

### 1.2 Fat tails and volatility clustering

In [ ]:
R.tails(df).round(3)

In [ ]:
R.fig_qq(df);

- **Fat tails, but only in the far tails.** Jarque-Bera rejects normality for 17 of 22 series. On average 3.4% of
  months lie beyond 2.33 sd (normal: 2%); the plastics PPI (excess kurtosis 8.9), Brent (7.0), nickel (5.1) and gas
  (3.9) are the heaviest. But only 16.0% of months lie beyond 1.28 sd (normal: 20%): the
  distributions are peaked in the middle. **Fat tails matter for P95 and the worst case (the buffer), not for an
  80% band.**
- **Volatility clustering** (ARCH-LM p < 0.05) in the steel chain (CRU CR / HDG / stainless, both steel PPIs, iron
  ore), the aluminium PPI, gas, power and both market-price cost indices. **Not** in the exchange-traded metals
  (copper, aluminium, nickel, zinc), Brent or EUR/USD: for those, a calm month says little about the next one.
- Monthly changes of the steel, aluminium and plastics PPIs and of CRU CR / HDG coil are strongly autocorrelated
  (0.6–0.8, last column of §1.3): these series trend, which is why ETS beats naive on the PPIs in the backtest.

### 1.3 Why the backtest's 80% bands are too narrow

Same set-up as the naive model of the backtest: band centred on today's price, width from past 3-month changes,
origins Jan-2020 to Jun-2026, only data known at each origin.

In [ ]:
b = R.band_coverage(df)
display(b.round(2))
b.drop(columns="n").mean().round(3)

- Coverage is **75% on average** (the backtest's 68% also includes the other models). The misses are **lopsided**:
  18% of outcomes end above the band and only 7% below (a correct band: 10% / 10%). 2020-26 was a rising-price
  period, and a band centred on today's price ignores that drift.
- The band is also **too narrow**: the sd of 3-month changes in 2020-26 is about **2× the sd known at the start of
  2020** (CRU CR / HDG coil and the plastics PPI more than 3×). With the realised sd (hindsight) coverage would be 81%.
- Using only the **last 24 months** does not help (72%): volatility rises only after a shock has started, and the
  window forgets the shock in the calm 2023-25 years, just before the 2025-26 moves.

**Conclusion for the forecasts**: the range must be (a) **wider** than recent history suggests, i.e. estimated on
a window that includes stress periods, or floored at the 2020-26 level; (b) **asymmetric**, using empirical
quantiles of past changes, not a symmetric normal band; (c) with **heavier tails** for P95 and beyond.

## 2. How much did the cost of one unit rise?

Every start month from Nov-2016, the change of the cost index over the next h months.

In [ ]:
r = R.cost_rises(df)
for c in ["median", "P90", "P95", "max", "share of windows > +10%"]:
    r[c] = pct(r[c])
r

In [ ]:
R.fig_rises(df);

In [ ]:
R.rise_contributions(prices, costs)  # percentage points of the rise coming from each material, top 3 windows

**Market-price indices** (what a buyer actually pays):

| | 3 months | 6 months | 9 months |
|---|---|---|---|
| RM6: P90 / P95 / max | +11% / +17% / +23% | +20% / +31% / +50% | +30% / +47% / +63% |
| AirSeT: P90 / P95 / max | +11% / +21% / +27% | +20% / +37% / +58% | +28% / +54% / +77% |

- The **PPI-proxy** indices understate this: 3-month P95 +11% (RM6) and +13% (AirSeT), maximum +17%.
- The largest rises all start in **late 2020 to early 2022** (reopening rally and war). In a typical window the cost
  hardly moves (median about 0%): a buffer is insurance against those episodes, not a trend.
- **Drivers**: AirSeT rises come mostly from steel (about 80%). Nov-2020 → Feb-2021 (+27%): CR coil +12 pp,
  stainless +9 pp, copper +5 pp. For RM6, copper is 53% of today's cost, but its largest rises came as much from
  **stainless and carbon steel** as from copper (Nov-2020 → Feb-2021: copper +7 pp, stainless +8 pp, carbon steel +6 pp).
- **For the quote rule**: a 3-month price validity with a buffer at the historical P90 would be about **+11%
  (RM6) / +11% (AirSeT)**, or +17% / +21% at P95. Both need the market-price index: the PPI proxy would suggest
  9% / 9% at P90.

## 3. Regimes

Boundaries set from turning points in the data: COVID low Apr-2020, reopening rally 2021, nickel squeeze Mar-2022,
energy price peak Aug-2022, normalisation 2023-25, copper rally from Oct-2025 (+48% to Sep-2026), oil shock Mar-2026.

In [ ]:
R.fig_regimes(df);

In [ ]:
vol, ratio, n = R.regime_vol(df)
display(n.rename("months"))
R.fig_regime_vol(ratio);

- **2022 was the extreme**: market-price cost indices 2.9× (RM6) and 3.6× (AirSeT) as volatile as in 2017-19; CR
  coil 4.1×, the steel PPIs 3.2–3.4×, gas 3.2×.
- **Volatility has not fully come back down**: in 2023-25 CR coil is still at 1.9× and the AirSeT index at 1.4× the
  2017-19 level. A risk model estimated on the calm years alone would be too narrow, which is what §1.3 found.
- **2025-26 is a trend, not turbulence**, for the BOM: copper rose 48%, yet the cost indices move at 0.85–1.2× their calm
  volatility, because the rise is steady. Volatility is high where the oil shock hits, the plastics PPI (5.5×)
  and Brent (1.8×), and in aluminium (1.7×). For RM6 the risk now is the **level and drift** of copper; for AirSeT
  (with 13% plastics and epoxy) the oil shock passing into resins.
- 2020 COVID has only 10 months, and the energy and steel series moved differently in it (energy collapsed, CRU
  steel was volatile), so its ratios are noisy.

## 4. How material prices move together

In [ ]:
full, calm, stress = R.correlations(df)
display(full.round(2))
full3, calm3, stress3 = R.correlations(df, h=3)
pd.DataFrame({"monthly changes": [R.mean_pairwise(m) for m in (full, calm, stress)],
              "3-month changes": [R.mean_pairwise(m) for m in (full3, calm3, stress3)]},
             index=["all months", "calm (2017-19, 2023-25)", "stress (2020-22)"]).round(2)

In [ ]:
R.fig_correlation(full, calm, stress, R.rolling_mean_corr(df));

In [ ]:
R.sum_of_quantiles(df, prices, costs).round(3)

- **Two blocks**: non-ferrous metals (copper–aluminium 0.58, nickel–both 0.48) and steel (CR–HDG 0.98, which is
  practically one price; CR–stainless 0.46). Across the blocks, correlations are low (0.14–0.26), and the plastics
  PPI sits in between (0.19–0.37).
- **Correlation rises in a crisis**: the average correlation of the six BOM materials is 0.22 in calm regimes and
  **0.45 in 2020-22** (3-month changes: 0.27 vs **0.62**). Rolling over 24 months it went from about 0.1 in 2020 to
  0.5 in 2021 and again in 2022, fell to below 0.1 in mid-2025 and is back near 0.3 in 2026.
- **Summing P90s is wrong, in both directions.** Over the whole sample, the cost-weighted sum of each material's P90
  overstates the P90 of the cost index by 21% (RM6) to 26% (AirSeT), and the P10 by 49–51%: materials do not all
  hit their extremes in the same quarter. But because correlations double in stress, this diversification is
  smallest exactly when it matters.

**Conclusion for the cost forecast**: the cost P10 / P90 must come from **joint price scenarios**, and those must
keep the stress-time correlation. Resampling **whole historical months (or blocks of months) for all materials
together** keeps the co-movement, the fat tails and the clustering at once. A single fixed correlation matrix
estimated on all months would understate the joint upside in a 2021-22-type episode.

## 5. Key material prices

The four prices that set most of the BOM cost, rebased to Jan-2017 = 100. Levels are EUR/t except the plastics PPI
(index). Stainless 304 ends Aug-2026 (ragged edge, see the data-quality notebook).

In [ ]:
E.fig_key_prices(prices, prices);
k = E.key_price_stats(prices, prices)
k[["change since Jan-17", "last vs peak"]] = pct(k[["change since Jan-17", "last vs peak"]])
k.round(1)

- **Copper is at a record**: €5,421 → €12,571/t since Jan-2017 (+132%), and Sep-2026 is the highest month of the
  sample. Most of the rise is recent: copper rose 48% from Oct-2025 (§3).
- **Steel is far below its 2022 peak**: CR coil €850/t is 42% below Apr-2022 (+30% since 2017), stainless 304
  €2,728/t is 49% below May-2022 (+11% since 2017). Both have been rising again since late 2025.
- **Plastics PPI** +38% since 2017, 13% below its Jun-2022 peak, with a jump in 2026 as the oil shock reached
  chemicals (§3: 5.5× its calm volatility in 2025-26).
- So the two products sit in different positions. **RM6** (53% copper) is quoted at the top of its main price: the risk
  runs both ways. **AirSeT** (50% steel, 36% copper) is split: its steel half is quoted well below the steel peak,
  and 2021-22 showed how fast steel can come back (+27% in 3 months for the AirSeT index, §2), while its copper
  third is at the record.

## 6. Material cost per unit over time

The cost index of §0 in EUR per unit: index × Sep-2026 cost per unit / 100. Solid = market prices, dashed = PPI
proxy. Only the BOM-priced raw materials are counted (AirSeT's "Other" 10% of mass, PE / PVC / PP, brass and the
unidentified remainder are not priced); plastics EUR/t are assumed; epoxy moves with the plastics PPI.

In [ ]:
E.fig_bom_index(panel);
u = E.unit_cost_stats(panel, costs)
u[["change 12 m", "last vs peak"]] = pct(u[["change 12 m", "last vs peak"]])
display(u.round(1))

uc = E.unit_cost(panel, costs)
gap = pd.DataFrame({"market prices": uc["RM6_NE_IQI_433520_mkt"] - uc["RMAirSeT_IIQ_24kV_mkt"],
                    "PPI proxy": uc["RM6_NE_IQI_433520"] - uc["RMAirSeT_IIQ_24kV"]})
gap.loc[["2025-09-01", "2026-09-01"]].rename(index=lambda m: f"{m:%b-%y}").rename_axis("RM6 - AirSeT (EUR / unit)").round(0)

- **Sep-2026 material cost per unit: RM6 €921, AirSeT €999** (both versions are equal at Sep-2026 by construction).
- **Last 12 months: RM6 +32%, AirSeT +28%** on market prices. The PPI proxy shows +27% / +21%: it understates both,
  AirSeT more, the same understatement as in §2.
- **Against the 2022 peak**: on market prices RM6 is 12% below its Apr-2022 peak (€1,046) and AirSeT 24% below
  (€1,311). On the PPI proxy RM6 is at its peak now and AirSeT 10% below its May-2022 peak.
- **AirSeT costs more than RM6**: about €80 more per unit in priced raw materials (€83 a year ago on market prices,
  €78 now). Copper rose for both products, so the gap barely moved.

## 7. Price risk by horizon

The typical move of each series: median of |price(t+h) / price(t) - 1| over every start month, Nov-2016 to Sep-2026.
This is the middle of the distribution; the tails that set the buffer are in §1.2 and §2.

In [ ]:
vol = E.volatility_by_horizon(panel)
E.fig_volatility(vol);
display(E.horizon_summary(vol))
(vol[9] / vol[3]).sort_values().round(2).rename("9 m / 3 m")

- **Over 3 months**: the EU PPIs typically move 2.1–3.8% and the four cost indices 2.8–4.3%, the exchange prices
  5.3–10.8%. The market-price indices move more than the PPI-proxy ones (RM6 4.3% vs 3.3%, AirSeT 4.3% vs 2.8%).
- **From 3 to 9 months the typical move grows about 1.8×** (median over all series), close to √3 = 1.73 for a
  random walk. The spread is wide:
  - The steel PPIs grow 2.8–2.9×, the PPI-proxy AirSeT index 2.2×. They trend (monthly changes autocorrelated
    0.6–0.8, §1.2), so the risk builds up faster than √h.
  - Copper in EUR, nickel, iron ore and the copper PPI grow only 1.2–1.3×: in this sample their typical moves partly
    reverse. Their tails still grow with the horizon (§2), so this does not make a 9-month quote safe.
- **Copper in EUR moves less than in USD at 9 months** (7.5% vs 11.4%, aluminium 10.0% vs 12.4%): over longer
  horizons the EUR/USD rate partly offsets the USD price. Schneider buys in EUR, so the EUR series are the right ones.
- **For the quote rule**: the buffer has to grow with the validity period, faster for the steel-heavy AirSeT.

## 8. Supply-chain transmission

Two measures per link. **next 3 m** = corr(last 3-month upstream change, next 3-month downstream change), no
overlap between the two windows. **same month** = corr of monthly changes. The chart shows "next 3 m" when it is at
least 0.3, otherwise "same month". Each pair uses its common months up to Sep-2026.

In [ ]:
G.fig_chain();
G.link_table().drop(columns=["upstream", "downstream"]).set_index("link").round(2)

- **Upstream prices move ahead** of the aluminium PPI (aluminium LME r = 0.59, gas 0.53, power DE 0.43), the
  stainless alloy surcharge (nickel 0.50), the steel PPI (gas 0.43) and the plastics PPI (Brent 0.43). Iron ore → CR
  coil (0.32) and zinc → HDG (0.30) are weak.
- **Same month only**: copper LME → copper PPI (0.93), CR → HDG coil (0.98, practically one price), benzene → epoxy
  (0.65), Brent → benzene (0.33). The data-quality notebook adds one more lead: CRU / MEPS CR coil leads the steel
  PPI by about one month (r ≈ 0.7).
- **What this means per product**: copper (53% of RM6, 36% of AirSeT) has no upstream signal, so RM6's main risk is pure exchange
  risk: P50 = today's price (or the LME forward), and the work goes into the range. For AirSeT the PPIs have
  drivers, but CR coil itself (29% of the market-price index) is only weakly led by iron ore.
- **Caveats**: about 40 independent 3-month windows, so the standard error of r is about 0.16: links at 0.3 are weak
  evidence, those at 0.5–0.6 stronger. These are in-sample correlations; the backtest has to confirm that adding
  the drivers improves the 3-month forecast.

## 9. Selling-price benchmark

Yearly P10 / P50 / P90 selling price of a 3-function RMU per brand, EUR ex-VAT
(`RMU_IQI_EU_Price_Benchmark_2017-2026_Synthetic.xlsx`, **synthetic**). According to the product specification
workbook in the data folder, every product is SF6-insulated except Eaton Xiria (air-insulated, vacuum switching), and
the ratings differ (Xiria 630 A / 21 kA, RM6 400 A / 12.5 kA): not like-for-like.

In [ ]:
bm = D.load_price_benchmark()
E.fig_price_benchmark(bm);
b = E.benchmark_table(bm)
cols = ["change 2026", "vs Schneider 2025", "vs Schneider 2026", "change 2017-25 (per year)"]
b[cols] = pct(b[cols])
display(b)

se = bm[bm.brand == "Schneider Electric"].set_index("year")["p50"]
rm6 = uc["RM6_NE_IQI_433520_mkt"]  # material cost per unit, market prices (§6)
pd.Series({"RM6 material cost Sep-26 (EUR)": rm6["2026-09-01"],
           "share of the 2026 P50 price": rm6["2026-09-01"] / se[2026],
           "material cost change Sep-25 -> Sep-26 (EUR)": rm6["2026-09-01"] - rm6["2025-09-01"],
           "P50 price change 2025 -> 2026 (EUR)": se[2026] - se[2025]}).round(3)

- **2017-25: slow drift.** SF6 brands +2.4–2.9% a year, Eaton +0.9% a year.
- **2026: the SF6 brands jump +32% to +40%** (Schneider €8,740 → €12,233, +40%), Eaton only +8%. Eaton went from 22%
  above Schneider in 2025 to 5.5% below in 2026: the SF6-free product lost its premium.
- **Material cost does not explain the jump.** RM6's priced raw materials are about €921, **≈ 8% of the 2026 P50
  price**, and rose by about €225 in 12 months, against a €3,500 rise in the selling price. A likely cause outside
  this dataset: the EU F-gas Regulation (EU) 2024/573 restricts new SF6 medium-voltage switchgear up to 24 kV from
  2026. To check before using it in the story.
- **For the quote rule**: in euros, the material risk is small next to the price. A 3-month P90 rise of RM6's cost
  index (+11.4%, §2) is about €105, under 1% of the P50 price. This counts only the BOM-priced raw materials, and the
  benchmark is synthetic, so this is an order of magnitude, not a margin calculation.

## 10. What this changes downstream

| Finding | Consequence |
|---|---|
| Levels have a unit root, changes are stationary | Forecast log changes; P10–P90 as changes from today's price |
| 80% bands miss mostly upwards; 2020-26 sd ≈ 2× earlier | Wider, asymmetric ranges from empirical quantiles of a window that includes 2020-22 |
| Fat tails beyond P95 | Buffer at P95 / worst case needs the empirical tail, not a normal one |
| PPI proxy halves the steel risk | Business layer (buffer, margin at risk) on the market-price index |
| 3-month buffer: P90 +11% / +11%, P95 +17% / +21% (RM6 / AirSeT) | Starting point for the quote rule |
| Correlation doubles in stress; summing quantiles is off by 21–51% | Joint scenarios by resampling whole months; never sum quantiles |
| Copper at a record (+132% since 2017), steel 42–49% below its 2022 peak | RM6 is quoted at the top of its main price, AirSeT half below the steel peak and a third at the copper record: different buffers per product |
| Cost per unit in 12 months: RM6 +32%, AirSeT +28% (market prices); PPI proxy +27% / +21% | Report cost per unit on market prices; the PPI proxy understates both moves |
| Typical move grows ~1.8× from 3 to 9 months; steel PPIs ~2.9× | Buffer grows with the price validity, faster for steel-heavy AirSeT |
| Upstream leads (r 0.4–0.6) for aluminium / steel / plastics PPIs and the stainless surcharge; none for copper | Drivers as candidate regressors for the PPIs (to confirm in the backtest); copper: P50 = today or LME forward |
| Priced raw materials ≈ 8% of the RMU selling price; 2026 SF6 price jump (+32–40%) not material-driven | Material buffer is small in EUR next to the price; SF6-free competitor now cheaper (synthetic benchmark) |